#### 연습 문제 
1. data 폴더 안에 rating_train.csv 파일을 로드 
2. 결측치를 제외
3. id 컬럼 제외 
4. 중복 데이터 제거 
4. label이 0인 데이터중 2000개를 추출 
5. label이 1인 데이터중 2000개를 추출 
6. 5번 6번의 결과를 단순 행결합
7. trian, test 데이터셋을 8:2 의 비율로 나눠준다 
8. tokenizer는 Okt를 사용 
9. 불필요한 품사를 제외 (사용할 품사 : 명사, 동사, 형용사, 부사. 파티클)
10. 글자 수의 제한은 2자리부터 가능 
11. tfidf를 사용하여 벡터화 
    - min_df = 2
    - ngram_range = (1,2)
12. 로지스틱회귀 모델을 사용하여 벡터화한 데이터에서 학습 random_state만 42로 고정
13. test데이터 셋을 이용하여 검증후 평가 지표 
14. 예측 결과, 원본의 데이터셋과 예측 확률을 하나의 데이터프레임으로 생성 
15. 결과물 제출 -> 평가 지표, 14번의 결과에서 상위 5개

In [1]:
import pandas as pd 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Okt

In [2]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [5]:
# 결측치 제거 
df.dropna(inplace = True)

In [7]:
# id 컬럼 제외 
df.drop('id', axis=1, inplace=True)

In [10]:
# 중복 데이터 제거 
df.drop_duplicates('document', inplace=True)

In [13]:
# label이 0인 데이터셋을 필터링 -> 2000개의 데이터를 추출 
df_0 = df.loc[df['label'] == 0, ]

In [18]:
df_0 = df_0.iloc[:2000]

In [19]:
df_1 = df.loc[df['label'] == 1].iloc[:2000]

In [ ]:
df_1

In [23]:
df2 = pd.concat( [df_0, df_1], axis=0 , ignore_index=True)

In [24]:
X = df2['document'].values
y = df2['label'].values

In [25]:
from sklearn.model_selection import train_test_split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [27]:
# 토크나이저 생성 
okt = Okt()

# 토큰화는 벡터화 작업에서 한번에 실행 시키기 위해 함수 선언 
def tokenize(text):
    # 품사 제한 : 명사, 동사, 형용사, 부사, 파티클
    allow_pos = ['Noun', 'Verb', 'Adjective', 'Adverb', 'KoreanParticle']
    # 문자의 길이 제한 2보다 크거나 같다. 
    len_word = 2
    result = []
    for word, pos in okt.pos(text):
        if (pos in allow_pos) & (len(word) >= len_word):
            result.append(word)
    return result

In [28]:
# 벡터화 class 생성 
tfidf_vec =  TfidfVectorizer(
    tokenizer=tokenize, 
    min_df = 2, 
    ngram_range= (1, 2)
)

In [29]:
# X의 train과 test를 토큰화 + 벡터화
X_train_vec = tfidf_vec.fit_transform(X_train)
X_test_vec = tfidf_vec.transform(X_test)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [30]:
# 생성된 피쳐의 개수가 몇개일까
print(len(tfidf_vec.get_feature_names_out()))

3715


In [31]:
logistic = LogisticRegression(random_state=42)

In [33]:
# 모델 학습 
logistic.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [34]:
pred = logistic.predict(X_test_vec)

In [35]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.76      0.80      0.78       400
           1       0.79      0.75      0.77       400

    accuracy                           0.77       800
   macro avg       0.77      0.77      0.77       800
weighted avg       0.77      0.77      0.77       800



In [36]:
# 예측의 값의 확률 
pred_proba = logistic.predict_proba(X_test_vec)

In [37]:
pred_proba

array([[0.14957659, 0.85042341],
       [0.32813575, 0.67186425],
       [0.94277907, 0.05722093],
       ...,
       [0.63915788, 0.36084212],
       [0.45398689, 0.54601311],
       [0.52468344, 0.47531656]], shape=(800, 2))

In [42]:
pd.DataFrame(list(
    zip(
        X_test, pred, pred_proba
    )
))

,0,1,2
0,2시간동안 우느라 지칠정도.. 대사하나하나 울게 만드네요 최고,1,"[0.14957658755923575, 0.8504234124407642]"
1,스파이더맨을 가장 좋아하는 1人~ 스파이더맨 완전 사랑합니다. 영원했으면 좋겠네요!!,1,"[0.32813574566237136, 0.6718642543376286]"
2,쓰레기중의 쓰레기 영화 모든것이 쓰레기다.,0,"[0.9427790735817955, 0.05722092641820451]"
3,이 편에 극장판 중에 제일 마음에 든다,1,"[0.3618111067826897, 0.6381888932173103]"
4,어렸을 때 봤을 때 영화 속 인어속 꼬리가 너무나 신비롭게 보였던.,1,"[0.4249641203248785, 0.5750358796751215]"
...,...,...,...
795,"발상은 좋았지만, B급영화의 한계를 보여줌. 그래도 재밌었음.",1,"[0.4792288102534511, 0.5207711897465489]"
796,보는내내 그대로 들어맞는 예측 카리스마 없는 악역,0,"[0.5025658482111444, 0.4974341517888556]"
797,"김수현작가 작품이라 보기시작했는데, 주인공들의 연기 뭔가 끓어당기는 맛이없네..",0,"[0.6391578837074933, 0.36084211629250673]"
798,"원주율 좀 만들어보자.,.,",1,"[0.453986888673364, 0.546013111326636]"


In [ ]:
# X_test, pred, pred_proba 3개의 데이터를 반복문을 통해서 반복 실행 -> 2차원으로 새로운 데이터를 구성 
data = []
for review, value, proba in zip(X_test, pred, pred_proba):
    # value는 1이라면 '긍정', 0이라면 '부정'
    value = '긍정' if value == 1 else '부정'
    # proba -> 둘중에 큰 값만 사용 -> 100을 곱한다. -> 소수점 3번째 자리에서 반올림 -> '%' 붙여준다
    proba = round(max(proba) * 100, 2)
    proba = str(proba) + '%'
    # review, value, proba 데이터를 하나의 리스트에 대입 
    data.append( [review, value, proba] )
data
    

In [44]:
pd.DataFrame(data, columns = ['review', 'Pred', 'Proba']).head(5)

,review,Pred,Proba
0,2시간동안 우느라 지칠정도.. 대사하나하나 울게 만드네요 최고,긍정,85.04%
1,스파이더맨을 가장 좋아하는 1人~ 스파이더맨 완전 사랑합니다. 영원했으면 좋겠네요!!,긍정,67.19%
2,쓰레기중의 쓰레기 영화 모든것이 쓰레기다.,부정,94.28%
3,이 편에 극장판 중에 제일 마음에 든다,긍정,63.82%
4,어렸을 때 봤을 때 영화 속 인어속 꼬리가 너무나 신비롭게 보였던.,긍정,57.5%


- 네이버 개발자센터를 이용해서 데이터를 수집
- 수집된 데이터를 이용하여 감정 평가 예측
    1. 네이버 개발자센터 접속 
    2. 서비스 api 신청 
    3. 서비스키를 이용해서 뉴스 데이터를 로드 
    4. 데이터를 이용하여 감정 분석

In [45]:
import os
from dotenv import load_dotenv
import requests
import re

In [46]:
load_dotenv()

True

In [47]:
naver_id = os.getenv('naver_api_id')
naver_secret = os.getenv('naver_api_secret')

In [48]:
naver_id

'X7M20vEn0Yl_rwCZqUYB'

In [54]:
# 네이버 api을 활용해서  news 제목들을 수집 
url = "https://openapi.naver.com/v1/search/news.json"

params = {
    'query' : '왕사남', 
    "display" : 30
}
headers = {
    'X-Naver-Client-Id' : naver_id, 
    'X-Naver-Client-Secret' : naver_secret
}

res = requests.get(
    url, 
    params = params, 
    headers=headers
)
res


<Response [200]>

In [ ]:
res.json()

1. res.json()에서 title부분의 value를 추출하여 하나의 리스트로 생성 
2. <b>, </b> 문자를 제거 
3. 위에서 만들어둔 벡터화를 이용하여 벡터화 작업 
4. 로지스틱 모델을 이용하여 예측 
5. 예측 값과 확률을 데이터프레임으로 생성 

In [64]:
new_titles = []
for item in res.json()['items']:
    # item -> dict 형태 데이터가 대입 
    # print(item['title'].replace('<b>', '').replace('</b>', ''))
    # break
    clean_title = re.sub(r'<[^>]*>', '', item['title'])
    # print(clean_title)
    new_titles.append(clean_title)

In [65]:
# 데이터를 벡터화 
X_api = tfidf_vec.transform(new_titles)

In [68]:
X_api.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(30, 3715))

In [69]:
pred_api = logistic.predict(X_api)
proba_api = logistic.predict_proba(X_api)

In [70]:
data = []
for title, value, proba in zip(new_titles, pred_api, proba_api):
    value = '긍정' if value == 1 else '부정'
    proba = round( max(proba) * 100, 2 )
    data.append( 
        {
            'title' : title, 
            'pred' : value, 
            'pred_proba' : proba
        }
    )

df_api = pd.DataFrame(data)

In [71]:
df_api.sort_values('pred_proba', ascending=False).head(10)

,title,pred,pred_proba
17,"연상호 감독 '군체', 400만 향해 돌진 중-'왕사남' 넘을까?",부정,90.52
28,'군체' 300만 관객 돌파…'왕사남'보다 빠르다,부정,68.53
27,"손익분기점 넘은 '군체', 310만 관객 돌파…'왕사남'보다 빠르다",부정,68.53
1,"칸이 열고 관객이 채웠다... '군체', 357만 돌파→'왕사남' 이어 흥행 성...",부정,64.52
24,"연간 2위 '군체', 350만명 최단기 기록…'왕사남'보다 2일 빨라",부정,63.45
29,"‘군체’, ‘왕사남’보다 빠르다…10일 만에 300만 손익분기점 돌파",부정,63.45
25,"“‘왕사남’보다 빠르다, 올해 개봉작 중 1등”…‘군체’ 310만 돌파",부정,62.69
3,'왕사남'이어 '살목지'도 흥행…4월 극장 매출 31.2%↑,부정,61.94
19,"'군체', 주말에만 97만명 봤다…흥행 장기 집권 기대",긍정,61.66
16,한강이 연 ‘소설의 시대’ 여전 …상반기 베스트셀러 1~3위 휩쓸어,긍정,61.28


In [72]:
df_api

,title,pred,pred_proba
0,‘왕사남’ 흥행에 문경새재 ‘구름인파’…올봄 관람객 153만명 몰렸다,부정,52.39
1,"칸이 열고 관객이 채웠다... '군체', 357만 돌파→'왕사남' 이어 흥행 성...",부정,64.52
2,‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다,부정,52.64
3,'왕사남'이어 '살목지'도 흥행…4월 극장 매출 31.2%↑,부정,61.94
4,단종 이어 안평대군…그들의 억울함은 이 시대 어떤 의미일까,부정,50.24
5,"밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행",부정,58.54
6,"'왕사남' 촬영지 문경새재, 올들어 153만명 찾았다",부정,59.31
7,'왕사남' 열풍에 찻사발축제 효과 '톡톡'…문경새재 방문객 153만 명 돌...,긍정,56.81
8,"반하다밀양 반값여행, 6월 얼음골 인증 이벤트로 열기 쭈~욱",부정,51.83
9,"'군체', 아시아 주요 지역 박스오피스 1위…글로벌 흥행 기대감",부정,54.94


#### 모델의 성능을 올려보자 
- 실제 모델의 성능 
    - 정확도 : 77% 
- 모델의 성능을 올릴 수 있는 방법
    - 데이터 양을 늘린다
    - 전처리 방법을 다른 방법으로 사용( 분석기, 벡터화 )
    - 스케일러를 이용 --> MaxAbs Scaler 사용
    - 벡터화 모델과 분류 모델의 파라미터 수정 
    - 모델을 변경 

In [73]:
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

In [74]:
ma_scaler = MaxAbsScaler()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = Pipeline(
    [
        (
            'vec', tfidf_vec
        ), 
        {
            'scaler', ma_scaler
        }, 
        (
            'model', logistic
        )
    ]
)

In [75]:
# 파라미터 조합 생성 
params = {
    'vec__min_df' : [2, 3], 
    'vec__ngram_range' : [ (1, 2), (1, 1) ], 
    'vec__max_features' : [ None, 1000 ], 
    'model__C' : [0.8, 0.9, 1.0], 
}

In [76]:
grid = GridSearchCV(
    estimator = pipe, 
    param_grid = params, 
    cv = cv, 
    verbose=1
)

In [ ]:
grid.fit(X, y)

In [79]:
grid.best_params_

{'model__C': 1.0,
 'vec__max_features': None,
 'vec__min_df': 2,
 'vec__ngram_range': (1, 1)}

In [81]:
pred_api = grid.predict(new_titles)
proba_api = grid.predict_proba(new_titles)

In [82]:
data = []
for title, value, proba in zip(new_titles, pred_api, proba_api):
    value = '긍정' if value == 1 else '부정'
    proba = round( max(proba) * 100, 2 )
    data.append( 
        {
            'title' : title, 
            'pred' : value, 
            'pred_proba' : proba
        }
    )

df_api = pd.DataFrame(data)

In [ ]:
df_api.head(10)

In [84]:
load_dotenv()

True

In [86]:
youtube_api = os.getenv('youtube_api')


- 유튜브에서 특정 영상의 댓글을 로드 
    - 구글 클라우드 콘솔에서 api를 신청 
    - 신청이 된 api key와 영상의 id 값이 필요 
    - 라이브러리 설치 
    - google-api-python-client  라이브러리 설치

In [88]:
# !pip install google-api-python-client

In [106]:
# 영상의 id를 하나 복사
# 유튜브 영상 url에서 가장 마지막 v=ID
video_id = 'z8BW9Fo9-wE'

In [107]:
from googleapiclient.discovery import build

In [108]:
youtube = build('youtube', 'v3', developerKey=youtube_api)

In [109]:
request = youtube.commentThreads().list(
    part = 'snippet', 
    videoId = video_id, 
    maxResults = 10, 
    textFormat = 'plainText'
)

In [110]:
res = request.execute()

In [111]:
from pprint import pprint

In [112]:
comment = []
for item in res['items']:
    comment.append(item['snippet']['topLevelComment']['snippet']['textDisplay'])
    # break

In [ ]:
comment

In [114]:
labels = [0, 0, 0, 0, 1, 1, 1, 0, 0, 1]
commnet_df = pd.DataFrame(zip(comment, labels), columns = ['review', 'label'])
commnet_df

,review,label
0,다른건 몰라도 이번건은 꼭 처벌하시고 넘어가셔야할듯 합니다.\n말씀하신대로 그냥 좋...,0
1,난간이 없는게 아니고 유리 난간으로 한거 같은데?,0
2,이야 동창생 잘되는게 싫어서 렉카를 끌어버리네? ㅋㅋㅋ,0
3,학폭이 맞아? 폭립 같은 걸 잘못 말 한 거 아니야?,0
4,얼굴보고 그러지마라.걍 잘 웃는 형이다,1
5,고소 인증 컨텐츠도 찍어주면 좋겠당ㅋㅋㅋ,1
6,캐치안했으면 기사뜨고 더 커졋을듯 ㅋㅋㅋ,1
7,우리 어어엄이 ㅠㅠㅠㅠ 유튜브각 뽑아주러 왔구나\n이제 고소빔 날리고.. 2차영상올...,0
8,어떻게 1일만에 100만회가 나오냐,0
9,아 그러니까 아카츠키 지망생이셨군요,1


In [115]:
pred = grid.predict(comment)
pred_proba = grid.predict_proba(comment)

In [116]:
pred

array([1, 0, 0, 1, 0, 1, 0, 1, 1, 0])

In [117]:
print(classification_report(pred, labels))

              precision    recall  f1-score   support

           0       0.33      0.40      0.36         5
           1       0.25      0.20      0.22         5

    accuracy                           0.30        10
   macro avg       0.29      0.30      0.29        10
weighted avg       0.29      0.30      0.29        10



In [118]:
data = []
for review, label, value, proba in zip(comment, labels, pred, pred_proba):
    label = '긍정' if label == 1 else '부정'
    value = '긍정' if value == 1 else '부정'
    proba = round( max(proba) * 100 , 2)
    proba = str(proba) + '%'

    data.append(
        {
            'review' : review, 
            'origin' : label, 
            'pred' : value, 
            'proba' : proba
        }
    )

df_youtube = pd.DataFrame(data)
df_youtube

,review,origin,pred,proba
0,다른건 몰라도 이번건은 꼭 처벌하시고 넘어가셔야할듯 합니다.\n말씀하신대로 그냥 좋...,부정,긍정,53.56%
1,난간이 없는게 아니고 유리 난간으로 한거 같은데?,부정,부정,68.64%
2,이야 동창생 잘되는게 싫어서 렉카를 끌어버리네? ㅋㅋㅋ,부정,부정,67.31%
3,학폭이 맞아? 폭립 같은 걸 잘못 말 한 거 아니야?,부정,긍정,54.42%
4,얼굴보고 그러지마라.걍 잘 웃는 형이다,긍정,부정,56.22%
5,고소 인증 컨텐츠도 찍어주면 좋겠당ㅋㅋㅋ,긍정,긍정,50.38%
6,캐치안했으면 기사뜨고 더 커졋을듯 ㅋㅋㅋ,긍정,부정,85.57%
7,우리 어어엄이 ㅠㅠㅠㅠ 유튜브각 뽑아주러 왔구나\n이제 고소빔 날리고.. 2차영상올...,부정,긍정,76.53%
8,어떻게 1일만에 100만회가 나오냐,부정,긍정,55.5%
9,아 그러니까 아카츠키 지망생이셨군요,긍정,부정,63.25%
